# 02 Data Cleaning

## Objective

This notebook creates the Silver data layer for the Olist Customer Analytics project.

The Silver layer contains cleaned and standardized tables generated from the original raw CSV files.

Main steps:

- Load raw datasets.
- Inspect structure and data quality.
- Convert date columns.
- Remove duplicated records.
- Enrich product categories with English translations.
- Create basic operational fields for delivery analysis.
- Save cleaned tables as Parquet files.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

## 2. Define paths

This notebook assumes it is located inside:

`01_customer_analytics/notebooks/`

Therefore, `..` points to the `01_customer_analytics/` project folder.

In [ ]:
PROJECT_PATH = Path("..")
RAW_PATH = PROJECT_PATH / "data" / "raw"
SILVER_PATH = PROJECT_PATH / "data" / "silver"

SILVER_PATH.mkdir(parents=True, exist_ok=True)

RAW_PATH, SILVER_PATH

## 3. Load raw datasets

In [ ]:
customers = pd.read_csv(RAW_PATH / "olist_customers_dataset.csv")
orders = pd.read_csv(RAW_PATH / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_PATH / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_PATH / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_PATH / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(RAW_PATH / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(RAW_PATH / "product_category_name_translation.csv")

## 4. Create table dictionary

This dictionary makes it easier to run quality checks across all datasets.

In [ ]:
tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation,
}

## 5. Inspect table structure

In [ ]:
summary = []

for table_name, df in tables.items():
    summary.append({
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicated_rows": df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)
summary_df

## 6. Inspect missing values

In [ ]:
for table_name, df in tables.items():
    print(f"\n{table_name.upper()}")
    missing_df = (
        df.isna()
        .sum()
        .reset_index()
        .rename(columns={"index": "column", 0: "missing_values"})
        .query("missing_values > 0")
        .sort_values("missing_values", ascending=False)
    )
    display(missing_df)

## 7. Convert date columns

Date columns are converted to datetime format so they can be used in delivery, retention, and time-series analysis.

In [ ]:
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

review_date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_date_cols:
    reviews[col] = pd.to_datetime(reviews[col], errors="coerce")

## 8. Remove duplicated rows

Only full-row duplicates are removed at this stage. Business-key deduplication will be handled later if needed.

In [ ]:
customers = customers.drop_duplicates()
orders = orders.drop_duplicates()
order_items = order_items.drop_duplicates()
payments = payments.drop_duplicates()
reviews = reviews.drop_duplicates()
products = products.drop_duplicates()
sellers = sellers.drop_duplicates()
geolocation = geolocation.drop_duplicates()
category_translation = category_translation.drop_duplicates()

## 9. Enrich product categories

Product categories are enriched with their English translation to make the analysis easier to read in dashboards and documentation.

In [ ]:
products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

## 10. Create operational delivery fields

These fields will support the delivery experience and customer satisfaction analyses.

In [ ]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

orders["is_delayed"] = np.where(
    orders["delivery_delay_days"] > 0,
    1,
    0
)

## 11. Refresh table dictionary after transformations

In [ ]:
silver_tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation,
}

silver_summary = []

for table_name, df in silver_tables.items():
    silver_summary.append({
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicated_rows": df.duplicated().sum()
    })

pd.DataFrame(silver_summary)

## 12. Save Silver tables

The cleaned datasets are saved as Parquet files in the Silver layer.

In [ ]:
customers.to_parquet(SILVER_PATH / "customers.parquet", index=False)
orders.to_parquet(SILVER_PATH / "orders.parquet", index=False)
order_items.to_parquet(SILVER_PATH / "order_items.parquet", index=False)
payments.to_parquet(SILVER_PATH / "payments.parquet", index=False)
reviews.to_parquet(SILVER_PATH / "reviews.parquet", index=False)
products.to_parquet(SILVER_PATH / "products.parquet", index=False)
sellers.to_parquet(SILVER_PATH / "sellers.parquet", index=False)
geolocation.to_parquet(SILVER_PATH / "geolocation.parquet", index=False)
category_translation.to_parquet(SILVER_PATH / "category_translation.parquet", index=False)

## 13. Validate Silver files

In [ ]:
silver_files = sorted(SILVER_PATH.glob("*.parquet"))

for file in silver_files:
    print(file.name)

## Conclusion

The Silver layer was created successfully.

The next step is to build the Gold analytical layer, including order-level facts, customer-level features, RFM metrics, and customer segments.